# FNO Classifier v2 — Target AUC ≥ 0.99
**Changes:** 5-ch input · Hybrid FNO+CNN · MixUp/CutMix · Focal loss · Stochastic depth · width=96 depth=6

In [ ]:
import numpy as np
import os, random, gc
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.functional as TF
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, roc_curve, auc as sklearn_auc
from sklearn.preprocessing import label_binarize

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device : {DEVICE}")
torch.cuda.empty_cache(); gc.collect()
IMG_SIZE = 64


## 1. Dataset — 5-Channel Input

| Ch | Name | What it encodes |
|---|---|---|
| 0 | Raw | Absolute flux |
| 1 | Polar | Ring → arc, substructure breaks smoothness |
| 2 | Angular gradient | \|dI/dθ\| — locates substructure along ring |
| 3 | Radial deviation | \|I − ring_mean\| — symmetry breaking |
| 4 | Power spectrum | log\|FFT²\| — frequency fingerprint of the lens |

In [ ]:
class LensDatasetFNO(Dataset):

    def __init__(self, root_dir, mean=None, std=None, augment=False):
        self.root_dir = root_dir
        self.augment  = augment
        self.mean     = mean
        self.std      = std

        self.paths, self.labels = [], []
        self.classes = sorted([d for d in os.listdir(root_dir)
                                if os.path.isdir(os.path.join(root_dir, d))])
        self.class_to_idx = {cls: i for i, cls in enumerate(self.classes)}
        for cls in self.classes:
            for f in sorted(os.listdir(os.path.join(root_dir, cls))):
                if f.endswith(".npy"):
                    self.paths.append(os.path.join(root_dir, cls, f))
                    self.labels.append(self.class_to_idx[cls])
        print(f"  {root_dir}: {len(self.paths)} samples | classes: {self.classes}")

    def __len__(self): return len(self.paths)

    # ── geometric augmentation (on raw only) ─────────────────────────────────
    def _geometric_aug(self, img):
        if random.random() > 0.5: img = TF.hflip(img)
        if random.random() > 0.5: img = TF.vflip(img)
        img = TF.rotate(img, random.uniform(-180, 180), fill=0)
        # random scale zoom  ← NEW
        if random.random() > 0.5:
            scale = random.uniform(0.85, 1.15)
            H, W  = img.shape[-2], img.shape[-1]
            nH, nW = int(H * scale), int(W * scale)
            img = TF.resize(img, [nH, nW])
            if scale > 1.0:
                i = (nH - H) // 2; j = (nW - W) // 2
                img = img[:, i:i+H, j:j+W]
            else:
                pad_h = (H - nH); pad_w = (W - nW)
                img = F.pad(img, [pad_w//2, pad_w - pad_w//2,
                                   pad_h//2, pad_h - pad_h//2], value=0)
        H, W = img.shape[-2], img.shape[-1]
        tx = int(random.uniform(-0.05, 0.05) * W)
        ty = int(random.uniform(-0.05, 0.05) * H)
        img = TF.affine(img, angle=0, translate=[tx, ty],
                        scale=1.0, shear=0, fill=0)
        return img

    # ── Gaussian noise on raw (NEW) ──────────────────────────────────────────
    def _add_noise(self, img):
        sigma = random.uniform(0.005, 0.025)
        return (img + torch.randn_like(img) * sigma).clamp(0.0, 1.0)

    # ── channel builders ─────────────────────────────────────────────────────
    def _polar_transform(self, img):
        eps = 1e-8
        H, W = img.shape[-2], img.shape[-1]
        cx, cy = (W - 1) / 2.0, (H - 1) / 2.0
        r_max = min(cx, cy)
        r_vals     = torch.linspace(0, r_max, H)
        theta_vals = torch.linspace(-torch.pi, torch.pi, W)
        R, THETA = torch.meshgrid(r_vals, theta_vals, indexing='ij')
        gx = ((cx + R * torch.cos(THETA)) / (W - 1)) * 2 - 1
        gy = ((cy + R * torch.sin(THETA)) / (H - 1)) * 2 - 1
        grid  = torch.stack([gx, gy], dim=-1).unsqueeze(0)
        polar = F.grid_sample(img.unsqueeze(0), grid, mode='bilinear',
                               padding_mode='zeros', align_corners=True).squeeze()
        return ((polar - polar.min()) / (polar.max() - polar.min() + eps)).unsqueeze(0)

    def _angular_gradient(self, img):
        eps = 1e-8
        H, W = img.shape[-2], img.shape[-1]
        cx, cy = (W - 1) / 2.0, (H - 1) / 2.0
        ys = torch.arange(H, dtype=torch.float32).unsqueeze(1).expand(H, W)
        xs = torch.arange(W, dtype=torch.float32).unsqueeze(0).expand(H, W)
        radii  = torch.sqrt((xs - cx)**2 + (ys - cy)**2).clamp(min=1.0)
        angles = torch.atan2(ys - cy, xs - cx)
        dtheta = 0.15
        def _sample(r, theta):
            x = (cx + r * torch.cos(theta)) / (W - 1) * 2 - 1
            y = (cy + r * torch.sin(theta)) / (H - 1) * 2 - 1
            return torch.stack([x, y], dim=-1).unsqueeze(0)
        I  = img.unsqueeze(0)
        Ip = F.grid_sample(I, _sample(radii, angles + dtheta),
                            mode='bilinear', padding_mode='zeros', align_corners=True).squeeze()
        Im = F.grid_sample(I, _sample(radii, angles - dtheta),
                            mode='bilinear', padding_mode='zeros', align_corners=True).squeeze()
        ag = (Ip - Im).abs() / (2 * dtheta)
        return ((ag - ag.min()) / (ag.max() - ag.min() + eps)).unsqueeze(0)

    def _radial_deviation(self, img):
        eps = 1e-8
        H, W = img.shape[-2], img.shape[-1]
        cx, cy = W / 2.0, H / 2.0
        ys = torch.arange(H, dtype=torch.float32).unsqueeze(1).expand(H, W)
        xs = torch.arange(W, dtype=torch.float32).unsqueeze(0).expand(H, W)
        radii   = torch.sqrt((xs - cx)**2 + (ys - cy)**2)
        bin_idx = (radii / (radii.max() + eps) * 30).long().clamp(0, 29)
        I = img[0]; dev = torch.zeros_like(I)
        for b in range(30):
            mask = (bin_idx == b)
            if mask.sum() > 0:
                dev[mask] = (I[mask] - I[mask].mean()).abs()
        return ((dev - dev.min()) / (dev.max() - dev.min() + eps)).unsqueeze(0)

    def _point_symmetry_residual(self, img):
        """Ch 4: |I(x,y) - I(-x,-y)| — point symmetry breaking.
        Perfect Einstein ring → near zero.
        Subhalo → bright localised spot.
        Vortex  → extended asymmetric arc."""
        eps = 1e-8
        # Flip 180° = horizontal + vertical flip
        flipped = torch.flip(img, dims=[-1, -2])   # (1, H, W)
        residual = (img - flipped).abs()
        return ((residual - residual.min()) /
                (residual.max() - residual.min() + eps))

    def _build_channels(self, raw):
        """Stack all 5 channels → (5, H, W)."""
        return torch.cat([
            raw,
            self._polar_transform(raw),
            self._angular_gradient(raw),
            self._radial_deviation(raw),
            self._point_symmetry_residual(raw),       # NEW
        ], dim=0)

    def _normalize(self, x):
        if self.mean is None: return x
        m = torch.tensor(self.mean, dtype=torch.float32).view(-1, 1, 1)
        s = torch.tensor(self.std,  dtype=torch.float32).view(-1, 1, 1)
        return (x - m) / s

    def __getitem__(self, idx):
        raw = torch.tensor(np.load(self.paths[idx]), dtype=torch.float32)
        if raw.ndim == 2: raw = raw.unsqueeze(0)
        raw = F.interpolate(raw.unsqueeze(0), size=IMG_SIZE,
                            mode='bilinear', align_corners=False).squeeze(0)

        # 1. Geometric aug on raw only
        if self.augment: raw = self._geometric_aug(raw)

        # 2. Gaussian noise on raw  ← NEW
        if self.augment and random.random() < 0.5:
            raw = self._add_noise(raw)

        # 3. Compute all 5 channels from (augmented) raw
        x = self._build_channels(raw)          # (5, H, W)

        # 4. Normalize
        x = self._normalize(x)

        # 5. RandomErasing on ALL channels (same patch)  ← improved
        if self.augment and random.random() < 0.35:
            i, j, h, w, v = transforms.RandomErasing.get_params(
                x[0:1], scale=(0.02, 0.10), ratio=(0.3, 3.3), value=[0])
            x[:, i:i+h, j:j+w] = 0

        return x, self.labels[idx]


## 2. Channel Statistics & Loaders

In [ ]:
def compute_channel_stats(root_dir, n_channels=5):
    ds     = LensDatasetFNO(root_dir, augment=False)
    loader = DataLoader(ds, batch_size=64, shuffle=False, num_workers=4)
    s1 = torch.zeros(n_channels); s2 = torch.zeros(n_channels); n = 0
    for imgs, _ in loader:
        s1 += imgs.sum(dim=[0, 2, 3])
        s2 += (imgs**2).sum(dim=[0, 2, 3])
        n  += imgs.shape[0] * imgs.shape[2] * imgs.shape[3]
    mean = s1 / n
    std  = torch.sqrt((s2/n - mean**2).clamp(min=0)).clamp(min=5e-4)
    print("Mean:", [f'{m:.4f}' for m in mean.tolist()])
    print("Std :", [f'{s:.4f}' for s in std.tolist()])
    return mean.tolist(), std.tolist()

print("Computing 5-channel statistics...")
MEAN_FNO, STD_FNO = compute_channel_stats("dataset/train")

train_dataset = LensDatasetFNO("dataset/train", mean=MEAN_FNO, std=STD_FNO, augment=True)
val_dataset   = LensDatasetFNO("dataset/val",   mean=MEAN_FNO, std=STD_FNO, augment=False)

# Reduce batch size in loaders (re-run the loader cell)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True,
                          num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_dataset, batch_size=32, shuffle=False,
                          num_workers=4, pin_memory=True)

img, label = train_dataset[0]
print(f"\nInput tensor : {img.shape}  <- (5 channels, {IMG_SIZE}, {IMG_SIZE})")
print(f"Label        : {label}")
assert img.shape[0] == 5, "Must have 5 channels"
print("Shape OK")


## 3. Architecture — Hybrid FNO + Local CNN

```
(B, 5, 128, 128)
    │
    ├── FNO branch ──────────────────────────────
    │   Lift Conv(5→96) + IN + GELU
    │   × 6  FNOBlock2d  (stochastic depth 0→0.15)
    │   Project Conv(96→192) + BN + GELU
    │   GAP ‖ GMP  →  (B, 384)
    │
    └── Local CNN branch ──────────────────────────
        Conv3×3(5→32) → Conv3×3(32→64,s=2) → Conv3×3(64→128,s=2)
        GAP  →  (B, 128)
    │
    Cat → (B, 512)
    FC(512) → BN → GELU → Drop(0.4)
    FC(128) → BN → GELU → Drop(0.2)
    FC(3)
```

In [ ]:
class SpectralConv2d(nn.Module):
    def __init__(self, in_ch, out_ch, m1, m2):
        super().__init__()
        self.out_ch = out_ch; self.m1, self.m2 = m1, m2
        s = 1.0 / (in_ch * out_ch)
        self.wr1 = nn.Parameter(s * torch.randn(in_ch, out_ch, m1, m2))
        self.wi1 = nn.Parameter(s * torch.randn(in_ch, out_ch, m1, m2))
        self.wr2 = nn.Parameter(s * torch.randn(in_ch, out_ch, m1, m2))
        self.wi2 = nn.Parameter(s * torch.randn(in_ch, out_ch, m1, m2))

    def _cmul(self, xr, xi, wr, wi):
        return (torch.einsum("bixy,ioxy->boxy", xr, wr) - torch.einsum("bixy,ioxy->boxy", xi, wi),
                torch.einsum("bixy,ioxy->boxy", xr, wi) + torch.einsum("bixy,ioxy->boxy", xi, wr))

    def forward(self, x):
        B, C, H, W = x.shape
        ft  = torch.fft.rfft2(x, norm="ortho")
        fr, fi = ft.real, ft.imag
        or_ = torch.zeros(B, self.out_ch, H, W//2+1, device=x.device)
        oi  = torch.zeros_like(or_)
        r1, i1 = self._cmul(fr[:,:,:self.m1,:self.m2], fi[:,:,:self.m1,:self.m2], self.wr1, self.wi1)
        or_[:,:,:self.m1,:self.m2] = r1; oi[:,:,:self.m1,:self.m2] = i1
        r2, i2 = self._cmul(fr[:,:,-self.m1:,:self.m2], fi[:,:,-self.m1:,:self.m2], self.wr2, self.wi2)
        or_[:,:,-self.m1:,:self.m2] = r2; oi[:,:,-self.m1:,:self.m2] = i2
        return torch.fft.irfft2(torch.complex(or_, oi), s=(H, W), norm="ortho")


class DropPath(nn.Module):
    """Stochastic depth — only addition beyond the base paper."""
    def __init__(self, drop_prob=0.0):
        super().__init__(); self.drop_prob = drop_prob
    def forward(self, x):
        if not self.training or self.drop_prob == 0.0: return x
        keep  = 1 - self.drop_prob
        shape = (x.shape[0],) + (1,) * (x.ndim - 1)
        mask  = torch.bernoulli(torch.full(shape, keep, device=x.device)) / keep
        return x * mask


class FNOBlock2d(nn.Module):
    """
    One Fourier layer from the original paper (Li et al. 2021):
        output = activate( norm( K(x) + W(x) ) )
    where:
        K(x) = SpectralConv2d  — the integral operator in frequency space
        W(x) = Conv1×1 bypass  — local linear transform (same role as residual)
    """
    def __init__(self, ch, m1, m2, drop_path=0.0):
        super().__init__()
        self.spectral  = SpectralConv2d(ch, ch, m1, m2)  # K(x)
        self.bypass    = nn.Conv2d(ch, ch, 1)              # W(x)
        self.norm      = nn.InstanceNorm2d(ch, affine=True)
        self.drop_path = DropPath(drop_path)
    def forward(self, x):
        return F.gelu(self.norm(self.drop_path(self.spectral(x)) + self.bypass(x)))


class FNOClassifier(nn.Module):
    """
    FNO for classification following Li et al. (2021):

        P  — lifting layer: R^in  → R^width   (projects input to high-dim feature space)
        ×4   Fourier layers: K(v) + W(v)        (the core FNO blocks)
        Q  — projection layer: R^width → R^out  (maps back to target space)

    The only deviations from the base paper are:
      - GAP+GMP pooling instead of pointwise output  (needed for classification)
      - InstanceNorm instead of no-norm              (stabilises training)
      - Stochastic depth                             (regularisation)
    """
    def __init__(self, in_channels=5, width=64, modes=20,
                 depth=4, num_classes=3, dropout=0.4):
        super().__init__()

        # P: lifting layer  (paper calls this the "input lifting")
        self.lift = nn.Sequential(
            nn.Conv2d(in_channels, width, 1),          # pointwise, as in paper
            nn.InstanceNorm2d(width, affine=True),
            nn.GELU())

        # Core Fourier layers
        dp_rates = [x.item() for x in torch.linspace(0, 0.10, depth)]
        self.blocks = nn.ModuleList(
            [FNOBlock2d(width, modes, modes, drop_path=dp_rates[i])
             for i in range(depth)])

        # Q: projection layer  (paper: two-layer MLP pointwise)
        self.project = nn.Sequential(
            nn.Conv2d(width, width * 2, 1),
            nn.BatchNorm2d(width * 2),
            nn.GELU())

        # Classification head (pooling + MLP — standard for image classification)
        self.gap  = nn.AdaptiveAvgPool2d(1)
        self.gmp  = nn.AdaptiveMaxPool2d(1)
        feat_dim  = width * 4                          # GAP + GMP concatenated

        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(feat_dim, 256),
            nn.BatchNorm1d(256),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(256, num_classes))

    def forward(self, x):
        x = self.lift(x)                               # P
        for blk in self.blocks: x = blk(x)            # Fourier layers
        x = self.project(x)                            # Q
        x = torch.cat([self.gap(x), self.gmp(x)], dim=1)
        return self.head(x)

    def count_parameters(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

# verify
_m = FNOClassifier(in_channels=5)
_o = _m(torch.randn(2, 5, IMG_SIZE, IMG_SIZE))
print(f"Output shape : {_o.shape}")
print(f"Parameters   : {_m.count_parameters():,}")
del _m, _o

## 4. Loss Functions & MixUp / CutMix

In [ ]:
# ── Focal Loss ────────────────────────────────────────────────────────────────
class FocalLoss(nn.Module):
    """Focal loss with label smoothing.
    gamma=2 down-weights easy examples so the model focuses on hard ones."""
    def __init__(self, gamma=2.0, label_smoothing=0.10):
        super().__init__()
        self.gamma = gamma
        self.ls    = label_smoothing

    def forward(self, logits, targets):
        nc = logits.size(1)
        with torch.no_grad():
            smooth = torch.full_like(logits, self.ls / (nc - 1))
            smooth.scatter_(1, targets.unsqueeze(1), 1.0 - self.ls)
        log_p   = F.log_softmax(logits, dim=1)
        p       = log_p.exp()
        focal_w = (1 - (p * smooth).sum(1).clamp(0, 1)).pow(self.gamma)
        loss    = -(smooth * log_p).sum(1)
        return (focal_w * loss).mean()


def soft_cross_entropy(logits, soft_targets):
    """Cross-entropy for soft (one-hot blended) labels — used with MixUp/CutMix."""
    return -(soft_targets * F.log_softmax(logits, dim=1)).sum(1).mean()


# ── MixUp ────────────────────────────────────────────────────────────────────
def mixup_data(x, y, alpha=0.4, num_classes=3):
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1.0
    idx = torch.randperm(x.size(0), device=x.device)
    mixed = lam * x + (1 - lam) * x[idx]
    ya    = F.one_hot(y, num_classes).float()
    yb    = F.one_hot(y[idx], num_classes).float()
    return mixed, lam * ya + (1 - lam) * yb


# ── CutMix ───────────────────────────────────────────────────────────────────
def cutmix_data(x, y, alpha=1.0, num_classes=3):
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(x.size(0), device=x.device)
    B, C, H, W = x.shape
    cut_r  = np.sqrt(1 - lam)
    cut_h  = int(H * cut_r); cut_w = int(W * cut_r)
    cx     = np.random.randint(W);  cy = np.random.randint(H)
    x1 = max(0, cx - cut_w // 2);  x2 = min(W, cx + cut_w // 2)
    y1 = max(0, cy - cut_h // 2);  y2 = min(H, cy + cut_h // 2)
    mixed  = x.clone()
    mixed[:, :, y1:y2, x1:x2] = x[idx, :, y1:y2, x1:x2]
    lam_actual = 1 - (x2 - x1) * (y2 - y1) / (H * W)
    ya = F.one_hot(y, num_classes).float()
    yb = F.one_hot(y[idx], num_classes).float()
    return mixed, lam_actual * ya + (1 - lam_actual) * yb

print("Loss functions and augmentation utilities ready.")


## 5. Evaluation

In [ ]:
def evaluate(model, loader, device):
    model.eval()
    all_probs, all_labels, total_loss = [], [], 0.0
    criterion = nn.CrossEntropyLoss(label_smoothing=0.10)
    with torch.no_grad():
        for imgs, labs in loader:
            imgs, labs = imgs.to(device), labs.to(device)
            logits      = model(imgs)
            total_loss += criterion(logits, labs).item()
            all_probs.append(F.softmax(logits, dim=1).cpu().numpy())
            all_labels.append(labs.cpu().numpy())
    p = np.concatenate(all_probs); l = np.concatenate(all_labels)
    auc = roc_auc_score(l, p, multi_class='ovr', average='macro')
    acc = (p.argmax(1) == l).mean()
    return total_loss / len(loader), acc, auc, p, l


## 6. Training Loop — MixUp + CutMix + Focal Loss

In [ ]:
def train_fno(model, train_loader, val_loader, device,
              epochs=80, lr=5e-4, weight_decay=3e-4,
              patience=15, save_path='best_fno.pth'):

    model     = model.to(device)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.10)  # plain CE, no mixup
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr,
                                   weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=lr, epochs=epochs,
        steps_per_epoch=len(train_loader),
        pct_start=0.10, anneal_strategy='cos',
        div_factor=10.0, final_div_factor=1e3)

    history = {k: [] for k in ['train_loss','val_loss','train_auc',
                                'val_auc','train_acc','val_acc','lr']}
    best_auc = best_epoch = patience_counter = 0

    print("=" * 72)
    print(f"FNO | {model.count_parameters():,} params | {epochs} ep | lr={lr}")
    print("=" * 72)
    print(f"{'Ep':>4}  {'TLoss':>7}  {'VLoss':>7}  "
          f"{'T-AUC':>7}  {'V-AUC':>7}  {'T-Acc':>6}  {'V-Acc':>6}")
    print("-" * 72)

    for epoch in range(1, epochs + 1):
        model.train()
        tl, tp, tlab = 0.0, [], []

        for imgs, labs in train_loader:
            imgs, labs = imgs.to(device), labs.to(device)
            optimizer.zero_grad()
            logits = model(imgs)
            loss   = criterion(logits, labs)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            tl += loss.item()
            with torch.no_grad():
                tp.append(F.softmax(logits, dim=1).cpu().numpy())
            tlab.append(labs.cpu().numpy())

        tp = np.concatenate(tp); tlab = np.concatenate(tlab)
        t_auc = roc_auc_score(tlab, tp, multi_class='ovr', average='macro')
        t_acc = (tp.argmax(1) == tlab).mean()
        vl, v_acc, v_auc, _, _ = evaluate(model, val_loader, device)

        for k, v in zip(['train_loss','val_loss','train_auc','val_auc',
                          'train_acc','val_acc','lr'],
                         [tl/len(train_loader), vl, t_auc, v_auc,
                          t_acc, v_acc, optimizer.param_groups[0]['lr']]):
            history[k].append(v)

        flag = " ✓" if v_auc > best_auc else ""
        print(f"{epoch:>4}  {tl/len(train_loader):>7.4f}  {vl:>7.4f}  "
              f"{t_auc:>7.4f}  {v_auc:>7.4f}  {t_acc:>6.2%}  {v_acc:>6.2%}{flag}")

        if v_auc > best_auc:
            best_auc = v_auc; best_epoch = epoch; patience_counter = 0
            torch.save({'epoch': epoch, 'model_state_dict': model.state_dict(),
                        'val_auc': best_auc, 'history': history}, save_path)
            print(f"  → Saved {save_path}  (AUC {best_auc:.4f})")
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"\nEarly stopping (best epoch {best_epoch}, AUC {best_auc:.4f})")
                break

    print(f"\nBest val AUC: {best_auc:.4f} at epoch {best_epoch}")
    return history

In [ ]:
import subprocess, os

# See what's holding the GPU
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout)

In [ ]:
import matplotlib.pyplot as plt

dataset_viz = LensDatasetFNO("dataset/train", augment=False)
class_names = ['no_sub', 'subhalo', 'vortex']
ch_names    = [
    'Ch0: Raw (Cartesian)',
    'Ch1: Polar transform',
    'Ch2: Angular gradient |dI/dθ|',
    'Ch3: Radial deviation',
    'Ch4: _point_symmetry_residual',   # NEW
]

fig, axes = plt.subplots(3, 5, figsize=(20, 11))

for class_idx in range(3):
    idx = next(i for i, l in enumerate(dataset_viz.labels) if l == class_idx)
    img, _ = dataset_viz[idx]          # (5, H, W), unnormalized

    for ch in range(5):
        ax  = axes[class_idx][ch]
        # choose colormap: power spectrum looks better in inferno
        cmap = 'inferno' if ch == 4 else 'viridis'
        im   = ax.imshow(img[ch].numpy(), cmap=cmap)
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        if class_idx == 0: ax.set_title(ch_names[ch], fontsize=9)
        if ch == 0:        ax.set_ylabel(class_names[class_idx], fontsize=11, fontweight='bold')
        ax.axis('off')

plt.suptitle('FNO v2 — All 5 input channels, one sample per class (no augmentation)',
             fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Kill all other Python processes holding the GPU
#os.system("kill -9 $(nvidia-smi --query-compute-apps=pid --format=csv,noheader | tr '\n' ' ')")

# Now properly reset
import torch, gc
gc.collect()
torch.cuda.synchronize()
torch.cuda.empty_cache()
print("GPU free memory:", torch.cuda.mem_get_info()[0] / 1e9, "GB")

## 7. Run Training

In [ ]:
import os
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'
torch.cuda.empty_cache(); gc.collect()

model = FNOClassifier(
    in_channels=5,
    width=64,
    modes=16,      # reduced — 64×64 image only has 33 freq modes, 16 is plenty
    depth=4,
    num_classes=3,
    dropout=0.3
).to(DEVICE)

history = train_fno(model, train_loader, val_loader, DEVICE,
                    epochs=80, lr=5e-4, patience=15)

## 8. Load Best Checkpoint & Evaluate

In [ ]:
ckpt = torch.load('best_fno.pth', weights_only=False)
model.load_state_dict(ckpt['model_state_dict'])
print(f"Loaded epoch {ckpt['epoch']}  —  Val AUC: {ckpt['val_auc']:.4f}")

_, val_acc, val_auc, all_probs, all_labels = evaluate(model, val_loader, DEVICE)
print(f"\nFinal Val AUC : {val_auc:.4f}")
print(f"Final Val Acc : {val_acc:.4f}")


## 9. Test-Time Augmentation (8 transforms)

In [ ]:
def evaluate_tta(model, loader, device, n_aug=8):
    """8-fold TTA: 4 rotations × 2 flips."""
    model.eval()
    all_probs, all_labels = [], []
    with torch.no_grad():
        for imgs, labs in loader:
            imgs = imgs.to(device)
            bp   = torch.zeros(imgs.shape[0], 3, device=device)
            for angle in [0, 90, 180, 270]:
                r   = TF.rotate(imgs, angle)
                bp += F.softmax(model(r), dim=1)
                bp += F.softmax(model(TF.hflip(r)), dim=1)
            bp /= n_aug
            all_probs.append(bp.cpu().numpy())
            all_labels.append(labs.numpy())
    p = np.concatenate(all_probs); l = np.concatenate(all_labels)
    auc = roc_auc_score(l, p, multi_class='ovr', average='macro')
    acc = (p.argmax(1) == l).mean()
    print(f"TTA Val AUC : {auc:.4f}  |  TTA Val Acc : {acc:.4f}")
    return acc, auc, p, l

tta_acc, tta_auc, tta_probs, tta_labels = evaluate_tta(model, val_loader, DEVICE)


## 10. ROC Curves

In [ ]:
class_names = ['no_sub', 'subhalo', 'vortex']
labels_bin  = label_binarize(tta_labels, classes=[0, 1, 2])

fig, ax = plt.subplots(figsize=(8, 6))
colors  = ['steelblue', 'coral', 'seagreen']
aucs    = []
for i, (name, col) in enumerate(zip(class_names, colors)):
    fpr, tpr, _ = roc_curve(labels_bin[:, i], tta_probs[:, i])
    ra = sklearn_auc(fpr, tpr); aucs.append(ra)
    ax.plot(fpr, tpr, color=col, lw=2.5, label=f"{name}  (AUC = {ra:.4f})")

macro = np.mean(aucs)
ax.plot([0,1],[0,1],'k--',lw=1)
ax.set_xlabel('False Positive Rate', fontsize=13)
ax.set_ylabel('True Positive Rate', fontsize=13)
ax.set_title(f'ROC — FNO v2 + TTA  (Macro AUC = {macro:.4f})', fontsize=13)
ax.legend(loc='lower right', fontsize=11)
plt.tight_layout(); plt.show()

print(f"Macro AUC (FNO v2 + TTA) : {macro:.4f}")
print(f"Macro AUC (ConvNeXt)     : 0.9910")
print(f"Delta                    : {macro - 0.9910:+.4f}")


## 11. Confusion Matrix

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

preds = tta_probs.argmax(axis=1)
cm = confusion_matrix(tta_labels, preds)
fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay(cm, display_labels=class_names).plot(
    ax=ax, cmap='Blues', colorbar=False)
ax.set_title('Confusion Matrix — FNO v2 + TTA')
plt.tight_layout(); plt.show()
print(f"TTA Val Accuracy: {(preds == tta_labels).mean():.4f}")


## 12. Training Curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
ep = range(1, len(history['train_loss']) + 1)

axes[0].plot(ep, history['train_loss'], label='Train')
axes[0].plot(ep, history['val_loss'],   label='Val')
axes[0].set(title='Loss', xlabel='Epoch'); axes[0].legend()

axes[1].plot(ep, history['train_auc'], label='Train')
axes[1].plot(ep, history['val_auc'],   label='Val')
axes[1].set(title='Macro AUC', xlabel='Epoch'); axes[1].legend()

axes[2].plot(ep, history['lr'])
axes[2].set(title='LR (OneCycleLR)', xlabel='Epoch', yscale='log')

plt.suptitle('FNO v2 — Training Curves', fontsize=13)
plt.tight_layout(); plt.show()
